In [ ]:
# =============================================================================
# FINAL PACKAGE — run in the notebook holding stage2_v3ce_output/ (Step 2 v3 notebook).
# Attach code_bundle.zip as a dataset. Writes /kaggle/working/<TEAM>_submission.zip
# =============================================================================
TEAM = "your_team_name"                                      # <- set your team name
FINAL_MATCHING = "stage2_v3ce_output/matching_results.tsv"   # <- the file that scored best on the leaderboard
FINAL_CANDIDATES = "stage2_v3ce_output/candidate_pairs.tsv"

import shutil, zipfile, importlib.metadata as md
from pathlib import Path
from IPython.display import FileLink, display
W, I = Path("/kaggle/working"), Path("/kaggle/input")
root = W / f"{TEAM}_submission"
shutil.rmtree(root, ignore_errors=True)
(root / "output").mkdir(parents=True)
shutil.copyfile(W / FINAL_MATCHING, root / "output" / "matching_results.tsv")
shutil.copyfile(W / FINAL_CANDIDATES, root / "output" / "candidate_pairs.tsv")
bundle = next(I.rglob("code_bundle.zip"), None)
if bundle is not None:
    zipfile.ZipFile(bundle).extractall(root)
else:                                                         # Kaggle may have unpacked it
    src = next(p.parent for p in I.rglob("Documentation_template.md") if (p.parent / "code").exists())
    shutil.copytree(src / "code", root / "code"); shutil.copyfile(src / "Documentation_template.md", root / "Documentation_template.md")
pkgs = ["numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "rapidfuzz", "pyarrow", "torch", "transformers",
        "sentence-transformers", "huggingface-hub", "indic-transliteration", "IndicTransToolkit", "tqdm", "psutil"]
lines = []
for p in pkgs:
    try: lines.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: lines.append(f"# {p}: installed at runtime by the scripts (pip)")
(root / "code" / "business_entity_resolution" / "requirements.txt").write_text("\n".join(lines) + "\n")
print(open(root / "code" / "business_entity_resolution" / "requirements.txt").read())
zpath = shutil.make_archive(str(W / f"{TEAM}_submission"), "zip", str(W), f"{TEAM}_submission")
for p in sorted(root.rglob("*")):
    if p.is_file() and p.parent.name in ("output", "business_entity_resolution", f"{TEAM}_submission"):
        print(f"  {str(p.relative_to(W)):<75} {p.stat().st_size/1e6:9.1f} MB")
print(f"\nZIP: {zpath}  {Path(zpath).stat().st_size/1e9:.2f} GB")
display(FileLink(Path(zpath).name))
